<a href="https://colab.research.google.com/github/wandb/examples/blob/master/colabs/pytorch/Simple_PyTorch_Integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<!--- @wandbcode{pytorch-video} -->

<img src="http://wandb.me/logo-im-png" width="400" alt="Weights & Biases" />

<!--- @wandbcode{pytorch-video} -->


# 🔥 = W&B ➕ PyTorch

This tutorial shows you how to integrate Weights & Biases (W&B) with PyTorch for experiment tracking, gradient monitoring, and model versioning. You can run this tutorial interactively here in Colab, or view the [complete documentation](https://docs.wandb.ai/models/tutorials/pytorch).

<div><img /></div>

<img src="https://wandb.me/mini-diagram" width="650" alt="Weights & Biases" />

<div><img /></div>

## What you'll learn

This tutorial demonstrates how to:

- Track hyperparameters and metrics during training
- Monitor model gradients and parameters
- Save and version trained models
- Visualize results in the W&B dashboard

**Time to complete**: Approximately 15 minutes

**Note**: Scroll to the bottom to see the complete training pipeline, or execute cells sequentially from top to bottom.

## Prerequisites

Before you begin, ensure you have:

- Python 3.7 or later
- PyTorch 1.9.0 or later installed ([installation guide](https://pytorch.org/get-started/locally/))
- A W&B account (sign up free at [wandb.ai](https://wandb.ai))
- Your W&B API key (find it at [wandb.ai/authorize](https://wandb.ai/authorize) after signing in)
- Experience training PyTorch models
- Understanding of machine learning concepts (epochs, hyperparameters, loss functions, and gradients)

## Quick start

For a quick copy-and-paste integration, add W&B to your PyTorch workflow with the `import wandb` statement and these four steps:

```python
import wandb

# 1. Initialize tracking
wandb.init(project="my-project", config={"learning_rate": 0.001, "epochs": 10})

# 2. Optional: Track model gradients
wandb.watch(model, log="all", log_freq=10)

# 3. Log metrics in your training loop
wandb.log({"loss": loss_value, "accuracy": acc_value})

# 4. Save your model
wandb.save("model.pth")
```

**Note**: The example above uses placeholder names. Replace with your own values:
- `"my-project"`: Your W&B project name
- `model`: Your PyTorch model instance
- `loss_value`, `acc_value`: Your computed metric values
- `"model.pth"`: Your model checkpoint filename

For a complete working example, see the full MNIST tutorial below or follow along with a [video tutorial](https://wandb.me/pytorch-video).

# Installation and Setup

The sections below provide a complete MNIST training example (handwritten digit classification) with W&B integration.

## Install W&B

Install the W&B library and ONNX using pip:

In [ ]:
!pip install wandb onnx -Uq

In [ ]:
import os
import random

import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from tqdm.auto import tqdm

# Ensure deterministic behavior
torch.backends.cudnn.deterministic = True
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Device configuration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

This code sets up the standard PyTorch environment. The key parts:
- **Random seeds**: Setting random seeds ensures reproducible results, which is important when comparing experiments tracked in W&B
- **Device configuration**: Determines whether to use GPU or CPU for training
- **Standard imports**: PyTorch, torchvision for the MNIST dataset, and tqdm for progress bars

## Import W&B and log in

Authenticate with W&B to enable data logging. 

When you run the next cells, you'll see a prompt asking you to paste your API key:

```
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:
```

Copy your API key from [wandb.ai/authorize](https://wandb.ai/authorize), paste it into the prompt, and press **Enter**. The key will be saved locally for future use.

In [ ]:
import wandb


In [ ]:
wandb.login()

# Initialize Experiment Tracking

## Configure hyperparameters

Store your hyperparameters in a configuration dictionary. This enables automatic logging and ensures consistency across your experiment:

In [ ]:
config = dict(
    epochs=5,
    classes=10,
    kernels=[16, 32],
    batch_size=128,
    learning_rate=0.005,
    dataset="MNIST",
    architecture="CNN")

Define the training pipeline with three main steps:

1. Create the model, data, and optimizer
2. Train the model
3. Evaluate performance

## Start a W&B run

Use `wandb.init()` to start tracking your experiment. This function establishes a connection to W&B servers and logs your configuration.

The `wandb.init()` context manager:
- Logs all hyperparameters from the `config` dictionary
- Ensures proper cleanup when training completes (automatically calls `wandb.finish()` at the end of the `with` block)
- Syncs data to W&B servers

**Note**: You can also call `wandb.init()` without a context manager (as shown in the quick start), but you'll need to manually call `wandb.finish()` at the end of your script. The context manager pattern is recommended as it handles cleanup automatically.

Use `wandb.config` to access hyperparameters throughout your code. This ensures the values W&B logs are exactly the same values used during training:


```python
# Good: Use wandb.config
optimizer = torch.optim.Adam(model.parameters(), lr=wandb.config.learning_rate)

# Avoid: Using original config dictionary
optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
```

In [ ]:
def make(config):
    # Make the data
    train, test = get_data(train=True), get_data(train=False)
    train_loader = make_loader(train, batch_size=config.batch_size)
    test_loader = make_loader(test, batch_size=config.batch_size)

    # Make the model
    model = ConvNet(config.kernels, config.classes).to(device)

    # Make the loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=config.learning_rate)
    
    return model, train_loader, test_loader, criterion, optimizer

# 📡 Define the Data Loading and Model

Define functions to load and prepare the MNIST dataset:

In [ ]:
def get_data(slice=5, train=True):
    """Load MNIST dataset.
    
    Args:
        slice: Use every nth example to reduce dataset size
        train: If True, load training set; otherwise load test set
        
    Returns:
        Subset of MNIST dataset
    """
    full_dataset = torchvision.datasets.MNIST(
        root=".",
        train=train,
        transform=transforms.ToTensor(),
        download=True
    )
    # Equivalent to slicing with [::slice]
    sub_dataset = torch.utils.data.Subset(
        full_dataset, indices=range(0, len(full_dataset), slice))
    
    return sub_dataset


def make_loader(dataset, batch_size):
    """Create DataLoader for dataset.
    
    Args:
        dataset: PyTorch dataset
        batch_size: Number of samples per batch
        
    Returns:
        DataLoader instance
    """
    loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=True,
        num_workers=2
    )
    return loader

# Note: This tutorial uses slice=5 to load only every 5th example from MNIST,
# reducing training time for demonstration purposes. In production, use the
# full dataset by removing the Subset wrapper or setting slice=1.

Create a convolutional neural network for image classification:

In [ ]:
# Convolutional neural network for MNIST classification

class ConvNet(nn.Module):
    """Convolutional neural network for MNIST classification."""
    
    def __init__(self, kernels, classes=10):
        super(ConvNet, self).__init__()
        
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, kernels[0], kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        self.layer2 = nn.Sequential(
            nn.Conv2d(kernels[0], kernels[1], kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        self.fc = nn.Linear(7 * 7 * kernels[-1], classes)
        
    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.reshape(out.size(0), -1)
        out = self.fc(out)
        return out

# Track Training Progress

Moving on in our `model_pipeline`, it's time to specify how we `train`.

Two `wandb` functions come into play here: `watch` and `log`.

## Monitor gradients and log metrics

Use `wandb.watch()` to automatically log gradients and parameters during training.

Parameters:
- `model`: Your PyTorch model
- `criterion`: Loss function (optional)
- `log`: What to track - `"gradients"`, `"parameters"`, `"all"`, or `None`
- `log_freq`: How often to log, in training batches (e.g., `10` means every 10 batches). Lower values provide more detail but increase logging overhead.

In [ ]:
def train(model, loader, criterion, optimizer, config):
    """Train the model and log metrics to W&B.
    
    Args:
        model: PyTorch model
        loader: Training data loader
        criterion: Loss function
        optimizer: Optimizer
        config: Configuration object
    """
    # Track gradients and parameters
    wandb.watch(model, criterion, log="all", log_freq=10)

    total_batches = len(loader) * config.epochs
    example_ct = 0  # Number of examples seen
    batch_ct = 0
    
    for epoch in tqdm(range(config.epochs)):
        for _, (images, labels) in enumerate(loader):
            loss = train_batch(images, labels, model, optimizer, criterion)
            example_ct += len(images)
            batch_ct += 1

            # Log loss and epoch metrics every 25 batches
            if ((batch_ct + 1) % 25) == 0:
                train_log(loss, example_ct, epoch)


def train_batch(images, labels, model, optimizer, criterion):
    """Execute a single training batch.
    
    Args:
        images: Batch of input images
        labels: Corresponding labels
        model: PyTorch model
        optimizer: Optimizer
        criterion: Loss function
        
    Returns:
        Loss value for the batch
    """
    images, labels = images.to(device), labels.to(device)
    
    # Forward pass
    outputs = model(images)
    loss = criterion(outputs, labels)
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()

    # Update weights
    optimizer.step()

    return loss

The `wandb.log()` function accepts a dictionary where:
- Keys are metric names (displayed in the dashboard)
- Values are the metric values to log
- The `step` parameter controls the x-axis in charts (here we use total examples seen, which provides a consistent metric across different batch sizes)

**Note**: This tutorial logs metrics every 25 batches (via `wandb.log()`) while gradients/parameters are logged every 10 batches (via `wandb.watch()`). These frequencies can be adjusted based on your needs—more frequent logging provides finer detail but increases overhead.

In [ ]:
def train_log(loss, example_ct, epoch):
    """Log training metrics to W&B.
    
    Args:
        loss: Current loss value
        example_ct: Number of examples processed
        epoch: Current epoch number
    """
    # Log metrics with example count as x-axis
    wandb.log({"epoch": epoch, "loss": loss}, step=example_ct)
    print(f"Loss after {str(example_ct).zfill(5)} examples: {loss:.3f}")

# Evaluate and Save Models

Evaluate your model on the test set and log results:

## Save model files

Use `wandb.save()` to upload model files to W&B.

**ONNX vs PyTorch checkpoints**: ONNX format (.onnx) provides cross-framework compatibility, allowing your model to run in different environments beyond PyTorch. PyTorch checkpoints (.pth) are faster to save and sufficient if you're staying within the PyTorch ecosystem. The example code tries ONNX first, falling back to PyTorch format if ONNX export fails.

In [ ]:
def test(model, test_loader):
    """Evaluate model on test set and log results.
    
    Args:
        model: Trained PyTorch model
        test_loader: Test data loader
    """
    model.eval()

    # Evaluate model
    with torch.no_grad():
        correct, total = 0, 0
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        print(f"Accuracy of the model on the {total} " +
              f"test images: {correct / total:%}")
        
        # Log test accuracy
        wandb.log({"test_accuracy": correct / total})

    # Save model in ONNX format (optional)
    try:
        torch.onnx.export(model, images, "model.onnx")
        wandb.save("model.onnx")
        print("Model exported to ONNX format successfully")
    except Exception as e:
        print(f"ONNX export skipped: {e}")
        # Save PyTorch checkpoint instead
        torch.save(model.state_dict(), "model.pth")
        wandb.save("model.pth")

In [ ]:
def model_pipeline(hyperparameters):
    """Complete training pipeline with W&B tracking.
    
    Args:
        hyperparameters: Configuration dictionary
        
    Returns:
        Trained model
    """
    # Initialize W&B run with context manager
    with wandb.init(project="pytorch-demo", config=hyperparameters):
        # Access hyperparameters through wandb.config
        config = wandb.config

        # Create model, data, and optimizer
        model, train_loader, test_loader, criterion, optimizer = make(config)
        print(model)

        # Train the model
        train(model, train_loader, criterion, optimizer, config)

        # Evaluate the model
        test(model, test_loader)

    return model

# Run the Training Pipeline

Execute the complete training pipeline. After running, W&B provides links to your project and run pages.

The run page includes:

1. **Charts**: Visualizes logged metrics, gradients, and parameters over time
2. **Overview**: Summary of run configuration, metadata, and hardware information
3. **Logs**: Shows console output from standard out during training
4. **Files**: Lists saved files, including `model.onnx` (viewable with the [Netron model viewer](https://github.com/lutzroeder/netron))
5. **Artifacts**: Tracks versioned datasets and model files (if using W&B Artifacts)

In [ ]:
# Build, train and analyze the model with the pipeline
model = model_pipeline(config)

# Conclusion

You've successfully integrated Weights & Biases with PyTorch! You now know how to track experiments, monitor training metrics and gradients, save model checkpoints, and visualize results in the W&B dashboard. These capabilities form the foundation for reproducible, collaborative machine learning workflows.

# Optimize Hyperparameters with Sweeps

W&B Sweeps automates hyperparameter optimization. Instead of manually testing different configurations, you can define a search space and let W&B explore it systematically.

Learn more in the [Sweeps documentation](https://docs.wandb.ai/guides/sweeps) or check out this [interactive Colab notebook](http://wandb.me/sweeps-colab) for a complete hyperparameter optimization tutorial.

<img src="https://imgur.com/UiQKg0L.png" alt="W&B Sweeps Dashboard showing parallel coordinates plot and performance metrics" />

# Next Steps

Now that you've integrated W&B with PyTorch, explore these additional features:

- [Environment variables](https://docs.wandb.ai/guides/track/environment-variables): Configure API keys for training on managed clusters
- [Offline mode](https://docs.wandb.ai/guides/track/public-api-guide): Train without internet access and sync results later
- [Self-managed deployment](https://docs.wandb.ai/guides/hosting): Install W&B on your own infrastructure
- [Sweeps](https://docs.wandb.ai/guides/sweeps): Advanced hyperparameter optimization strategies
- [Artifacts](https://docs.wandb.ai/guides/artifacts): Version and track datasets, models, and other files

# Example Gallery

Explore projects tracked with W&B in the [Gallery →](https://app.wandb.ai/gallery)